In [ ]:
import sympy as sp

def cdf_coeffs(n):
    # symbols
    y  = sp.symbols('y', real=True)
    xi = sp.symbols('xi')

    # parameters
    a  = sp.sqrt(2 * n + 3)

    # exact normalization - general formula for Nn
    # For Nn = ∫_{-a}^{a} (a²-y²)^n dy, using substitution y = a*sin(θ):
    # Nn = a^(2n+1) * ∫_{-π/2}^{π/2} cos^(2n+1)(θ) dθ
    # Using the beta function: Nn = a^(2n+1) * 2 * B(1/2, n+1) = a^(2n+1) * 2 * Γ(1/2)Γ(n+1)/Γ(n+3/2)
    # Since Γ(1/2) = √π and Γ(n+1) = n!, we get:
    # Nn = a^(2n+1) * 2 * √π * n! / Γ(n+3/2)
    # For integer n: Γ(n+3/2) = (n+1/2)(n-1/2)...(3/2)(1/2)√π = (2n+1)!!/(2^n) * √π
    # Therefore: Nn = a^(2n+1) * 2^(n+1) * n! / (2n+1)!!
    
    # Calculate using SymPy for verification
    Nn = sp.integrate((a**2 - y**2)**n, (y, -a, a))
    
    # General formula for Nn
    factorial_n = sp.factorial(n)
    double_factorial_2n_plus_1 = sp.factorial2(2*n + 1)
    Nn_general = a**(2*n + 1) * 2**(n + 1) * factorial_n / double_factorial_2n_plus_1
    
    print(f"n = {n}:")
    print(f"  Nn (computed) = {Nn}")
    print(f"  Nn (general)  = {Nn_general}")
    print(f"  Simplified    = {sp.simplify(Nn_general)}")
    print(f"  Verification  = {sp.simplify(Nn - Nn_general) == 0}")

    # PDF expanded around y = 0
    pdf = (a ** 2 - y ** 2) ** n / Nn
    pdf_series = sp.series(pdf, y, 0, 6*n + 2).removeO()  # keep plenty of terms

    # integrate series term-by-term from 0 to y  (ΔF = ∫0^y pdf)
    t = sp.symbols('t', real=True)
    deltaF = sp.integrate(pdf_series.subs(y, t), (t, 0, y))

    # rewrite in ξ = x/a  (x = ξ·a)
    deltaF_xi = sp.expand(deltaF.subs(y, xi * a))

    # coefficients up to ξ⁵
    c1 = deltaF_xi.coeff(xi, 1)
    c3 = deltaF_xi.coeff(xi, 3)
    c5 = deltaF_xi.coeff(xi, 5)

    p0, p1, p2 = sp.symbols('p0 p1 p2')
    sol = sp.solve(
        [
            sp.Eq(p0 + p1 + p2, c1),
            sp.Eq(-(p1 + 2 * p2), c3),
            sp.Eq(p2, c5),
        ],
        (p0, p1, p2)
    )
    sol = sol[0] if isinstance(sol, list) else sol

    return [sp.nsimplify(sol[p0]), sp.nsimplify(sol[p1]), sp.nsimplify(sol[p2])]

print("General formula for Nn:")
print("Nn = a^(2n+1) * 2^(n+1) * n! / (2n+1)!!")
print("where a = √(2n+3) and (2n+1)!! is the double factorial")
print()

for n in range(4):
    p0, p1, p2 = cdf_coeffs(n)
    print(f"n = {n}: p = [{p0}, {p1}, {p2}]   (float ≈ {[float(p0), float(p1), float(p2)]})")
    print()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Gaussian integral approximation function (same logic as in Allen)
def gauss_integral_approx(x, order_polynomial=2):
    a = np.sqrt(2 * order_polynomial + 3)
    xi = x / a
    eta = 1.0 - xi * xi
    p = [0.5, 0.25, 0.1875, 0.15625]  # corresponding to order=3

    # # Polynomial expansion
    # if order_polynomial < 3:
    #     raise ValueError("This approximation is only valid for order_polynomial=3")

    # Approximate cumulative distribution function (CDF approximation)
    # print(a)
    return 0.5 + xi * (p[0] + eta * (p[1] + eta * p[2]))

# x range (standard Gaussian is typically -5 to 5)
x = np.linspace(-5, 5, 500)

# True standard Gaussian CDF (scipy built-in)
true_cdf = norm.cdf(x)

# Approximate CDF
approx_cdf = np.array([gauss_integral_approx(xi, order_polynomial=2) for xi in x])

# Calculate error
error = approx_cdf - true_cdf

# Plot
plt.figure(figsize=(12, 6))

# Top plot: True vs Approximation
plt.subplot(1, 2, 1)
plt.plot(x, true_cdf, label="True Gaussian CDF", color="blue")
plt.plot(x, approx_cdf, label="Approx Gaussian CDF (order=3)", color="red", linestyle='--')
plt.xlabel("x")
plt.ylabel("CDF")
plt.legend()
plt.title("Gaussian CDF: True vs Approximation")
plt.axvline(2.6457513110645907, color="black", linestyle="--")
plt.axvline(-2.6457513110645907, color="black", linestyle="--")

# Bottom plot: Approximation error
plt.subplot(1, 2, 2)
plt.plot(x, error, label="Approximation Error", color="green")
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel("x")
plt.ylabel("Error (Approx - True)")
plt.legend()
plt.title("Approximation Error")


plt.tight_layout()
plt.show()


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def fast_erf(x):
    mod_x = abs(x)
    p = (
        1
        + 0.278393 * mod_x
        + 0.230389 * (x * x)
        + 0.000972 * mod_x * (x * x)
        + 0.078108 * (x * x) * (x * x)
    )
    return (1 if x >= 0 else -1) * (1 - 1.0 / ((p * p) * (p * p)))

# Test range: [-5, 5]
x_vals = np.linspace(-5, 5, 1000)
true_vals = [math.erf(x) for x in x_vals]
approx_vals = [fast_erf(x) for x in x_vals]
errors = np.abs(np.array(true_vals) - np.array(approx_vals))

# Output maximum and mean errors
max_error = np.max(errors)
mean_error = np.mean(errors)
print(f"Maximum error: {max_error:.6f}")
print(f"Mean error: {mean_error:.6f}")

# Comparison plot
plt.figure(figsize=(10, 6))
plt.plot(x_vals, true_vals, label='math.erf (true)', linewidth=2)
plt.plot(x_vals, approx_vals, '--', label='fast_erf (approximation)', linewidth=2)
plt.title("fast_erf vs math.erf comparison")
plt.xlabel("x")
plt.ylabel("erf(x)")
plt.legend()
plt.grid(True)
plt.show()

# Error plot
plt.figure(figsize=(10, 4))
plt.plot(x_vals, errors, label='Error |true - approx|', color='red')
plt.title("fast_erf approximation error")
plt.xlabel("x")
plt.ylabel("Absolute error")
plt.grid(True)
plt.legend()
plt.show()
